In [1]:
import pandas as pd
import numpy as np
import openml
from sklearn.preprocessing import StandardScaler,RobustScaler,LabelEncoder,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import numpy as np
import mlflow
from mlflow import client
from pathlib import Path

/Users/kingmopser/BachelorThesis/BachelorsThesisCode/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs

task = openml.tasks.get_task(task_ids[0])
dataset = task.get_dataset()
X, y, _, _ = dataset.get_data(target=task.target_name, dataset_format="dataframe")
task_ids

OpenMLServerError: Unexpected server error when calling https://www.openml.org/api/v1/xml/study/tabarena-v0.1. Please contact the developers!
Status code: 504
<html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx</center>
</body>
</html>


In [4]:
import socket, time
t = time.time()
socket.getaddrinfo("www.openml.org", 443)
print(time.time() - t)

0.005024909973144531


In [5]:
import time, openml

t = time.time()
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
print("get_suite:", time.time() - t)

task_ids = benchmark_suite.tasks

t = time.time()
task = openml.tasks.get_task(task_ids[0])
print("get_task:", time.time() - t)

t = time.time()
dataset = task.get_dataset()
print("get_dataset:", time.time() - t)

t = time.time()
X, y, _, _ = dataset.get_data(
    target=task.target_name,
    dataset_format="dataframe",
)
print("get_data:", time.time() - t)

OpenMLServerError: Unexpected server error when calling https://www.openml.org/api/v1/xml/study/tabarena-v0.1. Please contact the developers!
Status code: 504
<html>
<head><title>504 Gateway Time-out</title></head>
<body>
<center><h1>504 Gateway Time-out</h1></center>
<hr><center>nginx</center>
</body>
</html>


In [2]:
dfs=pd.read_csv("/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/data/task_metadata_tabarena51.csv")

characteristics=dfs.loc[dfs["name"].isin(["wine_quality", # perfect complexity
    "healthcare_insurance_expenses", # low complexity
     "Another-Dataset-on-used-Fiat-500", # too low complexity
     "miami_housing"]),:][["tid","name","NumberOfFeatures","target_feature","NumberOfFeatures","NumberOfInstances","NumberOfNumericFeatures","NumberOfSymbolicFeatures"]]
table_df_latex=characteristics.to_latex(caption="Dataset characterstics including shape and dimensions.")

with open("df_table.tex","w") as f:
    f.write(table_df_latex)

In [ ]:
table_df_latex

#### testing



In [ ]:
characteristics

#### loading all datasets


In [ ]:
#def preprocessing(dataset,):
benchmark_suite = openml.study.get_suite("tabarena-v0.1")
task_ids = benchmark_suite.tasks  # 51 task IDs
  
characteristics
tables = dict()
for id in characteristics["tid"]:
    task = openml.tasks.get_task(id)
    df = task.get_dataset()
    X, y, _, _ = df.get_data(target=task.target_name, dataset_format="dataframe")
    tables.update({df.name: {"X":X,"y":y.values.reshape(-1,1)}})
    

In [ ]:

def PreProcessing(name,data,test = 0.2,random_seed=0):
    
    X = data.get("X","")
    y = data.get("y","")
    
    is_binary = True if name == "QSAR-TID-11" else False
    is_housing = True if "month_sold" in X.columns else False
        
    if is_binary: # for QSAR TID 11 Dataset   
        print("correct detected")
        X = X.loc[:, X.nunique() > 1]
        
    if is_housing: 
        m = X["month_sold"].astype(int) - 1  # Jan=0 ... Dec=11
        X["month_sold_sin"] = np.sin(2 * np.pi * m / 12.0)
        X["month_sold_cos"] = np.cos(2 * np.pi * m / 12.0)
        X = X.drop(columns=["month_sold"])
                

    num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    
    #split into train and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test,random_state=random_seed)
    
        
    preprocessor = ColumnTransformer([("numeric",StandardScaler(),num_cols),
                             ("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)])
    
    yScaler = StandardScaler()
    
    X_train= pd.DataFrame(data=preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
    X_test = preprocessor.transform(X_test)
    
    y_train = yScaler.fit_transform(y_train)
    y_test = yScaler.transform(y_test)
    
    return X_train, X_test, y_train, y_test, yScaler

In [ ]:

X_train, X_test, y_train, y_test, yScaler = PreProcessing("healthcare_insurance_expenses",data=tables["healthcare_insurance_expenses"])
print(y_test)

In [ ]:
tables["healthcare_insurance_expenses"].get("y")

In [ ]:
Rng_Ho_Split = 12

for datasetname in tables:
        X_train_HO, X_test_HO, y_train_HO, y_test_HO, yScaler = PreProcessing(datasetname,data=tables[datasetname],random_seed=Rng_Ho_Split) # we do 1 hold-out-split
        # split train in to test and val
        X_train, X_test,y_train,y_test = train_test_split(X_train_HO,y_train_HO,random_state=Rng_Ho_Split)
        #hpo
        #create study
        #initialize child ruins
        #set main seed for sklearn. 
        def objecttive(trial):
            with ml
        



In [40]:
'''
helper functions for creating summary tables and plots
'''

from mlflow.tracking import MlflowClient
import matplotlib.pyplot as plt
from pathlib import Path
import mlflow
import pandas as pd


#mlflow db path 
MLFLOW_DB = "/Users/kingmopser/BachelorThesis/BachelorsThesisCode/src/data/mlflow.db"

CLIENT = MlflowClient(tracking_uri=f"sqlite:///{MLFLOW_DB}")
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

from mlflow.tracking import MlflowClient

def create_table(experiment_id, run_id,dataname):
    root_run_id = run_id

    root_run = mlflow.get_run(root_run_id)
    exp_id = root_run.info.experiment_id
    if experiment_id is not None and str(experiment_id) != str(exp_id):
        print(f"Warning: provided experiment_id={experiment_id} but root run belongs to experiment_id={exp_id}.")

    exp = mlflow.get_experiment(exp_id)
    experiment_name = exp.name if exp is not None else None

    seen = {root_run_id}
    frontier = [root_run_id]
    all_runs = []

    while frontier:
        pid = frontier.pop()
        kids = mlflow.search_runs(
            experiment_ids=[exp_id],
            filter_string=f"tags.mlflow.parentRunId = '{pid}'",
        )
        if kids.empty:
            continue
        all_runs.append(kids)
        for rid in kids["run_id"].tolist():
            if rid not in seen:
                seen.add(rid)
                frontier.append(rid)

    if not all_runs:
        return pd.DataFrame(), experiment_name

    runs = pd.concat(all_runs, ignore_index=True)

    parents = set(runs["tags.mlflow.parentRunId"].dropna())
    leaf_ids = sorted(set(runs["run_id"]) - parents)

    leaf_runs = runs[runs["run_id"].isin(leaf_ids)].copy()
    leaf_runs["experiment_name"] = experiment_name

    # Add parent run name + parent experiment name (direct parent of each leaf)
    client = MlflowClient()
    parent_ids = leaf_runs["tags.mlflow.parentRunId"].dropna().unique().tolist()

    parent_run_name = {}
    parent_experiment_name = {}
    for pid in parent_ids:
        prun = client.get_run(pid)
        parent_run_name[pid] = prun.data.tags.get("mlflow.runName")
        pexp = client.get_experiment(prun.info.experiment_id)
        parent_experiment_name[pid] = pexp.name if pexp is not None else None

    leaf_runs["parent_run_name"] = leaf_runs["tags.mlflow.parentRunId"].map(parent_run_name)
    leaf_runs["parent_experiment_name"] = leaf_runs["tags.mlflow.parentRunId"].map(parent_experiment_name)
    
    metrics= { 
    "metrics.RMSE": "RMSE",
    "metrics.Mean_Winkler_Score": "WinklerScore",
    "metrics.Negative_log_likelihood": "NLL",
    "metrics.Winkler_Coverage": "WinklerCoverage",
    "runtime": "Runtime (s)",}
    
    df = leaf_runs[["parent_run_name","metrics.RMSE",'metrics.Mean_Winkler_Score', 'metrics.Negative_log_likelihood','metrics.Winkler_Coverage','start_time',
       'end_time']]    
    #df = df.copy()
    #df = df.fillna(value=0)
    
    df["runtime"] = (pd.to_datetime(df["end_time"])-pd.to_datetime(df["start_time"])).dt.total_seconds()
    df["parent_run_name"] = df["parent_run_name"].apply(lambda x: "BDE" if x == "ModelBDE_robust" else "XGBoostLSS" if x == "ModelXGboostLSS_robust" else "TabIcL" if x == "ModelTabICL_robust" else "LinearRegression" if x =="ModelLG_robust" else "RandomForest")
    grouped_data =df.groupby("parent_run_name")[["metrics.RMSE",'metrics.Mean_Winkler_Score', 'metrics.Negative_log_likelihood','metrics.Winkler_Coverage',"runtime"]].agg(["mean","std"])

    def pm(cell):
        m = cell[("mean")]
        s = cell[("std")]
        if pd.isna(m):  # optional
            return "-"
        return f"{m:.4f} \\pm {s:.4f}"
    
    final_df = pd.DataFrame({"Model": grouped_data.index})
    for col, name in metrics.items():
        final_df[name] = grouped_data[col].apply(pm, axis=1).values

    latex = final_df.to_latex(index=False, escape=False)
    
    with open(f"metrics_{dataname}.tex","w") as f:
        f.write(latex)
    
    return df, experiment_name


In [42]:
df, exp_name = create_table("2","1b7fb705c8f1409588005c4f93b123e4","miami_housing")


print(df)

     parent_run_name  metrics.RMSE  metrics.Mean_Winkler_Score  \
5                BDE      0.274672                    0.743182   
6                BDE      0.279100                    0.791232   
7                BDE      0.277098                    0.785538   
8         XGBoostLSS      0.278372                    1.128762   
9         XGBoostLSS      0.284173                    1.116053   
10        XGBoostLSS      0.291426                    1.139012   
11            TabIcL  72738.928377               202633.750000   
12            TabIcL  72318.017742               206540.859375   
13            TabIcL  72715.088200               205055.625000   
14  LinearRegression      0.528780                         NaN   
15  LinearRegression      0.527926                         NaN   
16  LinearRegression      0.527717                         NaN   
17      RandomForest      0.288125                         NaN   
18      RandomForest      0.289620                         NaN   
19      Ra

/var/folders/z8/f4xxfstn37ndb5_8bsvv2v2w0000gn/T/ipykernel_12232/3640303839.py:87: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["runtime"] = (pd.to_datetime(df["end_time"])-pd.to_datetime(df["start_time"])).dt.total_seconds()
/var/folders/z8/f4xxfstn37ndb5_8bsvv2v2w0000gn/T/ipykernel_12232/3640303839.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["parent_run_name"] = df["parent_run_name"].apply(lambda x: "BDE" if x == "ModelBDE_robust" else "XGBoostLSS" if x == "ModelXGboostLSS_robust" else "Ta

In [3]:
df.fillna(value=0, inplace=True)

In [8]:
df["runtime"] = (pd.to_datetime(df["end_time"])-pd.to_datetime(df["start_time"])).dt.total_seconds()


In [25]:
df.groupby("parent_run_name")[["metrics.RMSE",'metrics.Mean_Winkler_Score', 'metrics.Negative_log_likelihood','metrics.Winkler_Coverage',"runtime"]].agg(['mean','std'])

TypeError: other must be a MultiIndex or a list of tuples

In [28]:
df

parent_run_name metrics.RMSE           metrics.Mean_Winkler_Score  \
                                  mean       std                       mean   
0         ModelBDE_robust     0.413263  0.030425                   1.946588   
1          ModelLG_robust     0.386924  0.000275                   0.000000   
2          ModelRF_robust     0.379667  0.002691                   0.000000   
3      ModelTabICL_robust   718.634684  5.927302                2896.923258   
4  ModelXGboostLSS_robust     0.388388  0.004992                   3.175385   

             metrics.Negative_log_likelihood               \
         std                            mean          std   
0   0.417500                        0.537112     0.209160   
1   0.000000                        0.469615     0.000844   
2   0.000000                        2.590779     0.133401   
3  13.864486                        0.000000     0.000000   
4   0.104679                     5477.188314  7519.839960   

  metrics.Winkler_Coverage              runtime            
                      mean       std       mean       std  
0                 0.935065  0.045105  37.489000  0.536993  
1                 0.000000  0.000000   2.698333  0.342993  
2                 0.000000  0.000000   3.105333  0.189540  
3                 0.908009  0.004959   5.771333  1.152267  
4                 0.503247  0.012987   5.150000  1.227244